# Agentic D&D Manager — Agent Implementation

In [14]:
import os
import json
import pickle
from datetime import datetime
from typing import Annotated

import pandas as pd

from haystack.components.agents import Agent
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function

## Session Management

In [15]:
SESSION_FILE = "session_data.pkl"


def initialize_empty_session() -> dict:
    return {
        "players": pd.DataFrame(columns=["name", "health", "defense", "damage", "origin", "status"]),
        "inventory": pd.DataFrame(columns=["player_name", "item_name", "description", "quantity", "is_consumable", "is_equippable"]),
        "status_effects": pd.DataFrame(columns=["player_name", "effect", "duration", "details"]),
        "skills": pd.DataFrame(columns=["player_name", "skill_name", "details", "cooldown", "cost"]),
        "npcs": pd.DataFrame(columns=["name", "job_title", "lore", "loot_table_json"]),
        "agent_notes": pd.DataFrame(columns=["timestamp", "event", "notes"]),
    }


def migrate_session(data: dict) -> dict:
    defaults = initialize_empty_session()
    for table, default_df in defaults.items():
        if table not in data:
            data[table] = default_df
        else:
            for col in default_df.columns:
                if col not in data[table].columns:
                    data[table][col] = None
    return data


def load_session(filepath: str = SESSION_FILE) -> dict:
    if os.path.exists(filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        return migrate_session(data)
    print(f"No session file found at '{filepath}'. Starting fresh.")
    return initialize_empty_session()


def save_session(filepath: str = SESSION_FILE) -> None:
    with open(filepath, "wb") as f:
        pickle.dump(SESSION, f)


SESSION = load_session()
print(f"Loaded session. Players: {list(SESSION['players']['name'])}")

Loaded session. Players: ['Alfred Winkleberry']


## Reference Data

| File | Purpose |
|---|---|
| `attributes.csv` | Valid player/character stat names |
| `materials.csv` | Valid armor materials + resistance stats |
| `body_coverage.csv` | Valid equip slots + coverage percentages |
| `prefixes.csv` | Valid item prefixes and their effects |
| `suffixes.csv` | Valid item suffixes and their effects |

In [16]:
REF_DIR = "."

REF = {
    "attributes":      pd.read_csv(f"{REF_DIR}/attributes.csv"),
    "materials": pd.read_csv(f"{REF_DIR}/materials.csv"),
    "body_coverage":   pd.read_csv(f"{REF_DIR}/body_coverage.csv"),
    "prefixes":        pd.read_csv(f"{REF_DIR}/prefixes.csv"),
    "suffixes":        pd.read_csv(f"{REF_DIR}/suffixes.csv"),
}

_VALID_ATTRIBUTES = ", ".join(REF["attributes"]["Attribute"].tolist())
_VALID_SLOTS = ", ".join(REF["body_coverage"]["Article"].tolist())

print("Reference data loaded:")
for name, df in REF.items():
    print(f"  {name}: {len(df)} rows")

Reference data loaded:
  attributes: 50 rows
  materials: 195 rows
  body_coverage: 24 rows
  prefixes: 117 rows
  suffixes: 138 rows


## LLM Setup

In [17]:
BASE_URL = "https://sdsu-rci-owui.nrp-nautilus.io/api"
API_KEY = open("./api_key.txt").read().strip()
ANSWER_MODEL = "gpt-oss:120b"

os.environ["OPENAI_API_KEY"] = API_KEY

llm = OpenAIChatGenerator(
    model=ANSWER_MODEL,
    api_base_url=BASE_URL,
    generation_kwargs={"temperature": 0.5},
)

## System Prompt

In [18]:
SYSTEM_PROMPT = f"""
You are a D&D session manager assistant. You help the Dungeon Master (DM) track and modify
session state. You do NOT narrate, adjudicate rules, or run combat. You manage data only.

## Your Tools — ONLY These Exist
You have exactly the following tools and nothing else. Never call any other tool name.
If you think you need a tool not in this list, you are wrong — use what is available or ask the DM.

| Tool | What it does |
|---|---|
| list_players | See all players and their status |
| query_player | Full details on one player (stats, inventory, effects, skills) |
| add_player | Add a new player to the roster |
| remove_player | Permanently delete a player (requires DM confirmation) |
| set_stat | Set a player stat to an exact value |
| change_stat | Increase or decrease a player stat by an amount |
| modify_inventory | Add or remove items from a player's inventory |
| modify_effects | Add, remove, or update status effects |
| modify_skills | Add, remove, or update skills |
| query_reference | Look up valid materials, slots, prefixes, suffixes, or attributes |
| generate_loot | Create a validated loot table (you write the items) |
| generate_npc | Create NPC data (name, job, lore, loot) |
| register_npc | Save a generated NPC to the session |
| agent_note | Log a bookkeeping note |

## How to Use query_reference
Parameters: table (required), filter_column (optional), filter_value (optional).
Valid table names: 'attributes', 'materials', 'body_coverage', 'prefixes', 'suffixes'
Examples:
- All prefixes:            query_reference(table='prefixes')
- One material's stats:    query_reference(table='materials', filter_column='Material', filter_value='Damascus Steel')
- One slot's coverage:     query_reference(table='body_coverage', filter_column='Article', filter_value='Chestplate')
- All suffixes:            query_reference(table='suffixes')
Never use 'key', 'name', or any other parameter name. Only 'table', 'filter_column', 'filter_value'.

## Valid Character Attributes
The only valid stat names in this campaign:
{_VALID_ATTRIBUTES}

## Valid Equipment Slots
The only valid slots an item can occupy:
{_VALID_SLOTS}

Requests referencing attributes or slots NOT in these lists must be clarified or rejected.
Always call query_reference before writing any item description that involves a material,
prefix, or suffix so the description is accurate and grounded in the reference data.

## Rules You Must Always Follow

1. NEVER GUESS. If any required information is missing, stop and ask the DM.
   - Adding a new item → you need: exact name, full description, is_consumable ('true'/'false'),
     is_equippable ('true'/'false'). Never invent these values.
   - Use query_reference to get real material stats and valid prefix/suffix effects before
     writing the item description. Do not make up resistance values or effect numbers.

2. CHECK PLAYER STATUS FIRST. Before modifying any player's data, call query_player to
   confirm their status. If they are dead, trapped, or incapacitated, ask the DM.

3. STAY IN SCOPE. Requests outside your tools should be politely declined:
   "That's outside my scope as a session manager."

4. OWNERSHIP. Items belong to the player in whose inventory they appear.
   To transfer from A to B: remove from A, add to B. Confirm with DM first.

5. ASK BEFORE ACTING ON AMBIGUITY. Never pick the most likely interpretation and proceed.

6. LOG KEY EVENTS. Use agent_note after significant events.

## Character Creation Flow
When the DM asks to create or generate a character:
1. If no name is given, ask for one before anything else.
2. Ask for: origin or class background, starting health, starting defense, starting damage.
   Do not silently default any of these — ask if not provided.
3. Optionally offer to set additional starting stats (mana, initiative, etc.) from the
   valid attributes list. If the DM says yes, walk through them or accept a batch.
4. Confirm all values with the DM before calling add_player.
"""

## Tools
### Query

In [19]:
def query_reference(
    table: Annotated[str, "Which reference table to query. One of: 'attributes', 'materials', 'body_coverage', 'prefixes', 'suffixes'."],
    filter_column: Annotated[str, "Column name to filter by. Pass empty string to return all rows."] = "",
    filter_value: Annotated[str, "Value to match in filter_column (case-insensitive). Only used when filter_column is set."] = "",
) -> dict:
    """
    Look up valid options from the campaign's reference data.

    Use this to:
    - Verify a material exists and get its stats (materials, filter by Material)
    - Check coverage percentage for an armor slot (body_coverage, filter by Article)
    - Find prefixes or suffixes for an item (prefixes or suffixes)
    - List all valid character attributes (attributes)

    Returns an error when a filtered value is not found — that means it is not valid in
    this campaign and should be clarified with the DM rather than used.
    """
    if table not in REF:
        return {"error": f"Unknown table '{table}'. Valid tables: {list(REF.keys())}"}

    df = REF[table].copy()

    if filter_column:
        if filter_column not in df.columns:
            return {"error": f"Column '{filter_column}' not in '{table}'. Available columns: {list(df.columns)}"}
        df = df[df[filter_column].astype(str).str.lower() == filter_value.lower()]
        if len(df) == 0:
            return {
                "error": (
                    f"No rows found where {filter_column}='{filter_value}' in '{table}'. "
                    "This value is not valid in this campaign — ask the DM to clarify."
                )
            }

    return {
        "table": table,
        "columns": list(df.columns),
        "rows": df.fillna("").to_dict(orient="records"),
        "count": len(df),
    }


def list_players() -> dict:
    """
    List every player in the session with their name, health, and status.
    Call this first when you need to know who exists before querying by name.
    """
    players = SESSION["players"]
    return {
        "players": players[["name", "health", "status"]].to_dict(orient="records"),
        "count": len(players),
    }


def query_player(
    player_name: Annotated[str, "Exact name of the player to look up."],
) -> dict:
    """
    Return all data for a specific player: stats, inventory, status effects, and skills.
    Use list_players first if you are unsure of the exact name.
    """
    players = SESSION["players"]
    player = players[players["name"] == player_name]
    if len(player) == 0:
        return {"error": f"Player '{player_name}' not found. Call list_players() to see all players."}

    return {
        "stats": player.iloc[0].to_dict(),
        "inventory": SESSION["inventory"][SESSION["inventory"]["player_name"] == player_name].to_dict(orient="records"),
        "status_effects": SESSION["status_effects"][SESSION["status_effects"]["player_name"] == player_name].to_dict(orient="records"),
        "skills": SESSION["skills"][SESSION["skills"]["player_name"] == player_name].to_dict(orient="records"),
    }


### Player CRUD (Create Read Update Delete )

In [20]:
def add_player(
    player_name: Annotated[str, "Player's name. Must be unique."],
    health: Annotated[int, "Starting health points."],
    defense: Annotated[int, "Defense stat."],
    damage: Annotated[int, "Damage stat."],
    origin: Annotated[str, "Player's origin, class, or one-line backstory."] = "",
) -> dict:
    """
    Add a new player to the session roster with their core starting stats.
    Fails if a player with the same name already exists. New players start with status='alive'.
    Only call this after confirming player_name, health, defense, damage, and origin with the DM.
    """
    if len(SESSION["players"][SESSION["players"]["name"] == player_name]) > 0:
        return {"error": f"Player '{player_name}' already exists."}

    new_row = pd.DataFrame([{
        "name": player_name, "health": health, "defense": defense,
        "damage": damage, "origin": origin, "status": "alive",
    }])
    SESSION["players"] = pd.concat([SESSION["players"], new_row], ignore_index=True)
    save_session()
    return {"success": True, "player": player_name, "stats": {"health": health, "defense": defense, "damage": damage}}


def remove_player(
    player_name: Annotated[str, "Exact name of the player to permanently remove."],
    confirm: Annotated[str, "Must be 'yes'. Never pass 'yes' without explicit DM confirmation in the chat."],
) -> dict:
    """
    Permanently delete a player and all their inventory, effects, and skills.
    Cannot be undone. Always ask the DM to explicitly confirm before calling with confirm='yes'.
    """
    if confirm.lower() != "yes":
        return {"error": "Removal not confirmed. Ask the DM explicitly, then pass confirm='yes'."}

    if len(SESSION["players"][SESSION["players"]["name"] == player_name]) == 0:
        return {"error": f"Player '{player_name}' not found."}

    for table in ["players", "inventory", "status_effects", "skills"]:
        key = "name" if table == "players" else "player_name"
        SESSION[table] = SESSION[table][SESSION[table][key] != player_name].reset_index(drop=True)

    save_session()
    return {"success": True, "removed": player_name}


### Stats

Previously one tool with `Optional[int]` params. Split into `set_stat` and `change_stat`
to avoid nullable types in the schema entirely.

In [21]:
def _get_player_stats(player_name: str) -> dict:
    row = SESSION["players"][SESSION["players"]["name"] == player_name]
    if len(row) == 0:
        return {}
    return row.iloc[0].to_dict()


def set_stat(
    player_name: Annotated[str, "Player's name."],
    stat: Annotated[str, "Stat to set: 'health', 'defense', or 'damage'."],
    new_value: Annotated[int, "The exact value to set the stat to."],
) -> dict:
    """
    Set a player's stat to an exact value. Use when you know the target number.
    Valid stats: health, defense, damage.
    """
    if stat not in ["health", "defense", "damage"]:
        return {"error": f"Unknown stat '{stat}'. Valid: health, defense, damage."}
    players = SESSION["players"]
    player = players[players["name"] == player_name]
    if len(player) == 0:
        return {"error": f"Player '{player_name}' not found."}
    idx = player.index[0]
    old = SESSION["players"].loc[idx, stat]
    SESSION["players"].loc[idx, stat] = new_value
    save_session()
    return {
        "success": True,
        "player": player_name,
        "stat": stat,
        "old_value": old,
        "new_value": new_value,
        "player_stats": _get_player_stats(player_name),
    }


def change_stat(
    player_name: Annotated[str, "Player's name."],
    stat: Annotated[str, "Stat to change: 'health', 'defense', or 'damage'."],
    amount: Annotated[int, "Amount to add or subtract. Positive adds, negative subtracts. Example: -5 reduces by 5."],
) -> dict:
    """
    Change a player's stat by a relative amount. Use when the DM says 'reduce by X' or 'add X'.
    Valid stats: health, defense, damage.
    """
    if stat not in ["health", "defense", "damage"]:
        return {"error": f"Unknown stat '{stat}'. Valid: health, defense, damage."}
    players = SESSION["players"]
    player = players[players["name"] == player_name]
    if len(player) == 0:
        return {"error": f"Player '{player_name}' not found."}
    idx = player.index[0]
    old = SESSION["players"].loc[idx, stat]
    new_value = old + amount
    SESSION["players"].loc[idx, stat] = new_value
    save_session()
    return {
        "success": True,
        "player": player_name,
        "stat": stat,
        "old_value": old,
        "new_value": new_value,
        "player_stats": _get_player_stats(player_name),
    }


### Modify Inventory

`is_consumable` and `is_equippable` are passed as the strings `'true'` or `'false'`
rather than booleans to avoid nullable schemas. Pass `'UNSET'` (default) when not applicable.

In [22]:
def modify_inventory(
    player_name: Annotated[str, "Player's name."],
    item_name: Annotated[str, "Name of the item to add or remove."],
    quantity_change: Annotated[int, "Positive to add, negative to remove. Must not be zero."],
    description: Annotated[str, "Required when adding a brand-new item. Include material, slot, prefix/suffix, and effect. Leave empty for existing items."] = "",
    is_consumable: Annotated[str, "Required for new items only. Pass 'true' if used up on use (e.g. potion), 'false' otherwise. Leave 'UNSET' for existing items."] = "UNSET",
    is_equippable: Annotated[str, "Required for new items only. Pass 'true' if it can be worn or wielded, 'false' otherwise. Leave 'UNSET' for existing items."] = "UNSET",
) -> dict:
    """
    Add or remove items in a player's inventory.

    For NEW items (not yet in inventory): description, is_consumable, and is_equippable
    are all required. Use query_reference to verify material and slot before writing
    the description. Never invent attribute names or material stats.

    For EXISTING items: only player_name, item_name, and quantity_change are needed.

    Returns an error if the player status is not 'alive' — ask the DM before proceeding.
    Returns an error if quantity_change is 0 — a no-op is always a mistake.
    """
    if quantity_change == 0:
        return {"error": "quantity_change cannot be 0. Pass a positive number to add or a negative number to remove."}

    players = SESSION["players"]
    player = players[players["name"] == player_name]
    if len(player) == 0:
        return {"error": f"Player '{player_name}' not found."}

    status = player.iloc[0].get("status", "alive")
    if status != "alive":
        return {"error": f"Player '{player_name}' has status '{status}'. Ask the DM whether to proceed."}

    inv = SESSION["inventory"]
    existing = inv[(inv["player_name"] == player_name) & (inv["item_name"] == item_name)]

    if quantity_change > 0:
        if len(existing) > 0:
            SESSION["inventory"].loc[existing.index[0], "quantity"] += quantity_change
        else:
            if not description or is_consumable == "UNSET" or is_equippable == "UNSET":
                return {
                    "error": (
                        f"'{item_name}' is new for {player_name}. Cannot add without "
                        "description, is_consumable ('true'/'false'), and is_equippable ('true'/'false'). "
                        "Ask the DM for these details. Use query_reference to verify material and slot first."
                    )
                }
            if is_consumable.lower() not in ("true", "false"):
                return {"error": "is_consumable must be 'true' or 'false'."}
            if is_equippable.lower() not in ("true", "false"):
                return {"error": "is_equippable must be 'true' or 'false'."}
            new_row = pd.DataFrame([{
                "player_name": player_name,
                "item_name": item_name,
                "description": description,
                "quantity": quantity_change,
                "is_consumable": is_consumable.lower() == "true",
                "is_equippable": is_equippable.lower() == "true",
            }])
            SESSION["inventory"] = pd.concat([inv, new_row], ignore_index=True)
    else:
        if len(existing) == 0:
            return {"error": f"Player '{player_name}' does not have '{item_name}'."}
        idx = existing.index[0]
        current = SESSION["inventory"].loc[idx, "quantity"]
        new_qty = current + quantity_change
        if new_qty < 0:
            return {"error": f"Cannot remove {abs(quantity_change)}x '{item_name}' — only {current} in inventory."}
        elif new_qty == 0:
            SESSION["inventory"] = SESSION["inventory"].drop(idx).reset_index(drop=True)
        else:
            SESSION["inventory"].loc[idx, "quantity"] = new_qty

    save_session()
    updated = SESSION["inventory"][SESSION["inventory"]["player_name"] == player_name]
    return {"success": True, "player": player_name, "inventory": updated.to_dict(orient="records")}


### Modify Effects & Skills

Optional `str` → `""` default. Optional `int` → `-1` default (game values are never negative).

In [23]:
def modify_effects(
    player_name: Annotated[str, "Player's name."],
    effect: Annotated[str, "Name of the status effect (e.g., 'poisoned', 'burning', 'blessed')."],
    action: Annotated[str, "One of: 'add', 'remove', 'update'."],
    duration: Annotated[int, "Duration in turns. Required when action='add'. Pass -1 when not applicable."] = -1,
    details: Annotated[str, "Description and mechanical impact of the effect. Required when action='add'. Leave empty when not applicable."] = "",
) -> dict:
    """
    Add, remove, or update a status effect on a player.
    'update' changes duration and/or details on an existing effect.
    When action='add', both duration (>= 0) and details are required.
    When action='update', at least one of duration (>= 0) or details must be provided.
    """
    if len(SESSION["players"][SESSION["players"]["name"] == player_name]) == 0:
        return {"error": f"Player '{player_name}' not found."}
    if action not in ["add", "remove", "update"]:
        return {"error": "action must be 'add', 'remove', or 'update'."}

    effects = SESSION["status_effects"]
    existing = effects[(effects["player_name"] == player_name) & (effects["effect"] == effect)]

    if action == "add":
        if duration < 0 or not details:
            return {"error": "Adding an effect requires duration (>= 0) and details."}
        if len(existing) > 0:
            return {"error": f"Effect '{effect}' already on {player_name}. Use action='update'."}
        new_row = pd.DataFrame([{"player_name": player_name, "effect": effect, "duration": duration, "details": details}])
        SESSION["status_effects"] = pd.concat([effects, new_row], ignore_index=True)

    elif action == "remove":
        if len(existing) == 0:
            return {"error": f"Player '{player_name}' does not have effect '{effect}'."}
        SESSION["status_effects"] = effects.drop(existing.index).reset_index(drop=True)

    elif action == "update":
        if len(existing) == 0:
            return {"error": f"Effect '{effect}' not found on {player_name}. Use action='add' first."}
        if duration < 0 and not details:
            return {"error": "update requires at least one of: duration (>= 0) or details."}
        idx = existing.index[0]
        if duration >= 0:
            SESSION["status_effects"].loc[idx, "duration"] = duration
        if details:
            SESSION["status_effects"].loc[idx, "details"] = details

    save_session()
    updated = SESSION["status_effects"][SESSION["status_effects"]["player_name"] == player_name]
    return {"success": True, "player": player_name, "status_effects": updated.to_dict(orient="records")}


def modify_skills(
    player_name: Annotated[str, "Player's name."],
    skill_name: Annotated[str, "Name of the skill."],
    action: Annotated[str, "One of: 'add', 'remove', 'update'."],
    details: Annotated[str, "What the skill does mechanically. Required when action='add'. Leave empty when not applicable."] = "",
    cooldown: Annotated[int, "Cooldown in turns. Required when action='add'. Pass -1 when not applicable."] = -1,
    cost: Annotated[str, "Resource cost, e.g. '10 mana' or 'none'. Required when action='add'. Leave empty when not applicable."] = "",
) -> dict:
    """
    Add, remove, or update a skill for a player.
    When action='add', details, cooldown (>= 0), and cost are all required.
    When action='update', at least one of details, cooldown (>= 0), or cost must be provided.
    """
    if len(SESSION["players"][SESSION["players"]["name"] == player_name]) == 0:
        return {"error": f"Player '{player_name}' not found."}
    if action not in ["add", "remove", "update"]:
        return {"error": "action must be 'add', 'remove', or 'update'."}

    skills = SESSION["skills"]
    existing = skills[(skills["player_name"] == player_name) & (skills["skill_name"] == skill_name)]

    if action == "add":
        if not details or cooldown < 0 or not cost:
            return {"error": "Adding a skill requires details, cooldown (>= 0), and cost."}
        if len(existing) > 0:
            return {"error": f"Skill '{skill_name}' already exists for {player_name}. Use action='update'."}
        new_row = pd.DataFrame([{
            "player_name": player_name, "skill_name": skill_name,
            "details": details, "cooldown": cooldown, "cost": cost,
        }])
        SESSION["skills"] = pd.concat([skills, new_row], ignore_index=True)

    elif action == "remove":
        if len(existing) == 0:
            return {"error": f"Player '{player_name}' does not have skill '{skill_name}'."}
        SESSION["skills"] = skills.drop(existing.index).reset_index(drop=True)

    elif action == "update":
        if len(existing) == 0:
            return {"error": f"Skill '{skill_name}' not found for {player_name}. Use action='add' first."}
        if not details and cooldown < 0 and not cost:
            return {"error": "update requires at least one of: details, cooldown (>= 0), or cost."}
        idx = existing.index[0]
        if details:
            SESSION["skills"].loc[idx, "details"] = details
        if cooldown >= 0:
            SESSION["skills"].loc[idx, "cooldown"] = cooldown
        if cost:
            SESSION["skills"].loc[idx, "cost"] = cost

    save_session()
    updated = SESSION["skills"][SESSION["skills"]["player_name"] == player_name]
    return {"success": True, "player": player_name, "skills": updated.to_dict(orient="records")}


### Generate Loot & NPCs

In [24]:
def generate_loot(
    items_json: Annotated[
        str,
        'JSON array of loot items you have generated. '
        'Each item must have: {"name": str, "description": str, "is_equippable": bool}. '
        'Use query_reference to verify materials and slots before writing descriptions.',
    ],
    context: Annotated[str, "What the loot is from, e.g. 'goblin chieftain chest'."] = "",
) -> dict:
    """
    Validate a loot table you have generated and return it as a clean structured list.
    After DM approval, use modify_inventory to distribute items to players.
    """
    try:
        items = json.loads(items_json)
    except json.JSONDecodeError as e:
        return {"error": f"items_json is not valid JSON: {e}"}
    if not isinstance(items, list):
        return {"error": "items_json must be a JSON array."}

    validated = []
    for i, item in enumerate(items):
        missing = [k for k in ["name", "description", "is_equippable"] if k not in item]
        if missing:
            return {"error": f"Item {i} ('{item.get('name', '?')}') is missing fields: {missing}"}
        validated.append({
            "name": str(item["name"]),
            "description": str(item["description"]),
            "is_equippable": bool(item["is_equippable"]),
        })

    return {
        "context": context,
        "loot_table": validated,
        "item_count": len(validated),
        "next_step": "Present to DM, then use modify_inventory to assign approved items.",
    }


def generate_npc(
    name: Annotated[str, "NPC's full name."],
    job_title: Annotated[str, "NPC's role or profession."],
    lore: Annotated[str, "One to three sentences of background."],
    loot_table_json: Annotated[str, 'JSON array of items. Each: {"name": str, "description": str, "is_equippable": bool}.'],
) -> dict:
    """
    Generate and return NPC data: name, job title, lore, and loot table.
    Non-destructive — returns data only. Call register_npc to save.
    """
    try:
        loot = json.loads(loot_table_json)
    except json.JSONDecodeError as e:
        return {"error": f"loot_table_json is not valid JSON: {e}"}

    validated = []
    for i, item in enumerate(loot):
        missing = [k for k in ["name", "description", "is_equippable"] if k not in item]
        if missing:
            return {"error": f"Loot item {i} is missing fields: {missing}"}
        validated.append({"name": str(item["name"]), "description": str(item["description"]), "is_equippable": bool(item["is_equippable"])})

    return {
        "name": name, "job_title": job_title, "lore": lore, "loot_table": validated,
        "next_step": "Present to DM. If approved, call register_npc to save.",
    }


def register_npc(
    name: Annotated[str, "NPC's name."],
    job_title: Annotated[str, "NPC's job title."],
    lore: Annotated[str, "NPC's background lore."],
    loot_table_json: Annotated[str, "JSON array of the NPC's loot table."],
) -> dict:
    """
    Save a generated NPC to the session. Only call after DM approval.
    Fails if an NPC with the same name is already registered.
    """
    npcs = SESSION["npcs"]
    if len(npcs[npcs["name"] == name]) > 0:
        return {"error": f"NPC '{name}' is already registered."}
    new_row = pd.DataFrame([{"name": name, "job_title": job_title, "lore": lore, "loot_table_json": loot_table_json}])
    SESSION["npcs"] = pd.concat([npcs, new_row], ignore_index=True)
    save_session()
    return {"success": True, "npc": name}


def agent_note(
    event: Annotated[str, "Short label, e.g. 'players entered dungeon', 'met NPC Gruul'."],
    notes: Annotated[str, "Full details: what happened, who was involved, outcomes."],
) -> dict:
    """Log a bookkeeping note to the session event log with an automatic timestamp."""
    new_row = pd.DataFrame([{"timestamp": datetime.now().isoformat(), "event": event, "notes": notes}])
    SESSION["agent_notes"] = pd.concat([SESSION["agent_notes"], new_row], ignore_index=True)
    save_session()
    return {"success": True, "logged": event}

## Wire Up the Agent

In [25]:
tools = [
    create_tool_from_function(query_reference),
    create_tool_from_function(list_players),
    create_tool_from_function(query_player),
    create_tool_from_function(add_player),
    create_tool_from_function(remove_player),
    create_tool_from_function(set_stat),
    create_tool_from_function(change_stat),
    create_tool_from_function(modify_inventory),
    create_tool_from_function(modify_effects),
    create_tool_from_function(modify_skills),
    create_tool_from_function(generate_loot),
    create_tool_from_function(generate_npc),
    create_tool_from_function(register_npc),
    create_tool_from_function(agent_note),
]

# Haystack 2.x uses chat_generator. If TypeError, try Agent(llm=llm, ...) for older builds.
agent = Agent(
    chat_generator=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Agent ready with {len(tools)} tools.")

Agent ready with 14 tools.


## Chat Loop

Conversation history is accumulated across turns so the agent remembers everything said
earlier in the same session.

| Command | Effect |
|---|---|
| `!save` | Save session to disk immediately |
| `!reload` | Re-read `session_data.pkl` into memory (picks up characters added elsewhere) |
| `!clear` | Wipe conversation history only — session data is untouched |
| `!quit` | Save and exit |

In [26]:
conversation_history = []

print("D&D Manager ready.  Commands: !save  !reload  !clear  !quit\n")

while True:
    user_input = input("DM: ").strip()

    if not user_input:
        continue

    if user_input.startswith("!"):
        cmd = user_input.lower()
        if cmd in ("!quit", "!exit"):
            save_session()
            print("Session saved. Goodbye.")
            break
        elif cmd == "!save":
            save_session()
            print("Saved.\n")
        elif cmd == "!reload":
            SESSION.update(load_session())
            print(f"Reloaded. Players: {list(SESSION['players']['name'])}\n")
        elif cmd == "!clear":
            conversation_history = []
            print("Conversation history cleared. Session data unchanged.\n")
        else:
            print(f"Unknown command '{user_input}'. Available: !save  !reload  !clear  !quit\n")
        continue

    # Echo the user's message back so both sides of the conversation are visible
    print(f"> {user_input}")

    conversation_history.append(ChatMessage.from_user(user_input))

    try:
        result = agent.run(messages=conversation_history)
        new_messages = result["messages"]
        conversation_history.extend(new_messages)

        # Strip tool-role messages from history after every turn.
        # The SDSU API returns 400 when tool call/result messages appear in a subsequent
        # request. This is safe because all state lives in SESSION — the agent can always
        # re-query instead of relying on remembered tool output.
        conversation_history = [
            msg for msg in conversation_history
            if msg.role.value in ("user", "assistant") and msg.text
        ]

        print(f"\nAgent: {new_messages[-1].text}\n")

    except Exception as e:
        # Discard the user message so a broken turn doesn't poison future requests
        conversation_history.pop()
        print(f"\n[Turn failed — message removed from history. Try rephrasing or use !clear.]\n")

D&D Manager ready.  Commands: !save  !reload  !clear  !quit

> Remove all players

Agent: I can see which players are currently in the session and then, with your explicit confirmation, remove them.  

Would you like me to list the current players first, and then confirm removal of each (or all) of them?

> no just remove all

Agent: Sure thing. To permanently delete the player(s), I need a clear confirmation. Please reply with **confirm='yes'** to remove all current players (currently Alfred Winkleberry).

> yes

Agent: All player characters have been permanently removed from the session, and the action has been logged.

> list all players

Agent: There are currently **no players** in the session.

Conversation history cleared. Session data unchanged.

> create a new character named Timothy Quickfingers who's a bard from Eldenwood Villa

Agent: Sure! I can add Timothy Quickfingers to the party.

I have his name and background (bard from Eldenwood Villa).  
To create his character shee

## Example Conversation #1
D&D Manager ready.  Commands: !save  !reload  !clear  !quit

> Hello can you load any characters?

Agent: Here’s the current roster of characters in the session:

- **Alfred Winkleberry** – Health 20, Status alive  

Let me know if you’d like to view details for Alfred, add a new character, or make any changes to his stats, inventory, effects, or skills.

> let's view Alfred's inventory

Agent: Alfred Winkleberry’s inventory is currently empty. Let me know if you’d like to add items, view his other details, or make any changes.

> Let's give him a Lyre, made of Thornwood, give it the prefix "Warped" and the Suffix "of the Kraken"

Agent: Alfred Winkleberry now has **1 × Lyre** in his inventory:

- **Name:** Lyre  
- **Description:** *Warped Lyre of the Kraken, made of Thornwood (30% Pierce Resistance, 40% Bludgeon Resistance, 28% Slash Resistance, Defense Score 24). Effect: -4% Defense & -4% Armor Class; Passive: +8% Vitality & +6% Tenacity; On‑Hit: 20% chance to Entangle target (cannot move for 1 turn).*  
- **Equippable:** Yes  
- **Consumable:** No  

Let me know if you’d like to equip it, add more items, modify stats, or anything else.

> List out your functions

Agent: Here are the functions (tools) I have available to manage the session state:

| Function | What it does |
|---|---|
| **list_players** | Returns a list of every player in the session with their name, health, and status. |
| **query_player** | Returns full details for a specific player: stats, inventory, status effects, and skills. |
| **add_player** | Adds a new player to the roster with starting health, defense, damage, and origin. |
| **remove_player** | Permanently deletes a player and all of their data (requires explicit DM confirmation). |
| **set_stat** | Sets a player’s stat (health, defense, or damage) to an exact value. |
| **change_stat** | Adjusts a player’s stat by a relative amount (positive or negative). |
| **modify_inventory** | Adds or removes items in a player’s inventory (new items require full description, consumable flag, and equippable flag). |
| **modify_effects** | Adds, removes, or updates status effects on a player. |
| **modify_skills** | Adds, removes, or updates a player’s skill. |
| **query_reference** | Looks up campaign reference data (armor materials, body‑coverage percentages, prefixes, suffixes, or attribute list). |
| **generate_loot** | Validates a loot table you’ve created and returns it as a clean, structured list. |
| **generate_npc** | Generates NPC data (name, job title, lore, and loot table) for review. |
| **register_npc** | Saves a previously generated NPC to the session (after DM approval). |
| **agent_note** | Logs a bookkeeping note to the session event log with an automatic timestamp. |

These are the only actions I can perform on the game state. Let me know which one you’d like to use!

> let's query Alfred Winkleberry

Agent: **Alfred Winkleberry – Full Details**

- **Stats**
  - Health: 20  
  - Defense: 1  
  - Damage: 1  
  - Origin: bard class  
  - Status: alive  

- **Inventory**
  1. **Lyre** (quantity 1) – *Warped Lyre of the Kraken, made of Thornwood (30% Pierce Resistance, 40% Bludgeon Resistance, 28% Slash Resistance, Defense Score 24). Effect: -4% Defense & -4% Armor Class; Passive: +8% Vitality & +6% Tenacity; On‑Hit: 20% chance to Entangle target (cannot move for 1 turn).*  
     - Equippable: Yes  
     - Consumable: No  

- **Status Effects:** None  
- **Skills:** None  

Let me know if you’d like to modify anything (add/remove items, change stats, apply effects, etc.).

> Let's add a skill to him, give Alfred "Dizzy" as a status effect and write "Makes Alfred miss 10% of the time" as the description

Agent: I can add the “Dizzy” effect, but I need to know how long it should last. Could you tell me the duration in turns (or use ‑1 for a permanent effect)?

> Nice work on realizing I meant status effect. It should last 2 turns

[Turn failed — message removed from history. Try rephrasing or use !clear.]

Saved.

Reloaded. Players: ['Alfred Winkleberry']

> give Alfred "Dizzy" for 2 turns which reduces his accuracy by 10%

[Turn failed — message removed from history. Try rephrasing or use !clear.]

Conversation history cleared. Session data unchanged.

> Give alfred "dizzy" for 2 turns which should reduce his accuracy by 10%

Agent: The effect name “dizzy” is fine, but the campaign’s list of valid attributes does not include an **accuracy** attribute. Could you let me know which valid attribute you’d like the “dizzy” effect to affect (e.g., **Attack**, **Critical Scale**, **Hit Chance**, etc.)? Once I have the correct attribute, I’ll add the “dizzy” status effect to Alfred for 2 turns.

> nice catch, make it affect Initiative negatively by 10%

Agent: I couldn’t find a player named **Alfred** in the current session. Would you like me to create a new character named Alfred (you’ll need to provide origin/class, starting health, defense, and damage), or is Alfred already in the party under a different spelling? Let me know how you’d like to proceed.

> list all players

Agent: The **dizzy** status effect has been added to **Alfred Winkleberry** for 2 turns, reducing his Initiative by 10%.  

*Event logged.*

> list all players

Agent: Here are the players currently in the session:

| Name                | Health | Status |
|---------------------|--------|--------|
| Alfred Winkleberry  | 20     | alive |

> give Alfred Winkleberry the status effect "Dizzy" which negatively affects initiative by 10% for 2 turns

Agent: Here’s the current roster:

| Name               | Health | Status |
|--------------------|--------|--------|
| Alfred Winkleberry | 20     | alive  |

**Status effects:** Alfred already has the “dizzy” effect applied, which reduces his Initiative by 10 % for 2 turns (duration = 2). No further action is needed unless you’d like to extend the duration or apply a different effect.

> okay cool, can you generate an npc? Including her name, job title, lore, and loot table.

[Turn failed — message removed from history. Try rephrasing or use !clear.]

Conversation history cleared. Session data unchanged.

> show me everything about Alfred Winkleberry

[Turn failed — message removed from history. Try rephrasing or use !clear.]

Session saved. Goodbye.

## Example Conversation #2
D&D Manager ready.  Commands: !save  !reload  !clear  !quit

> Remove all players

Agent: I can see which players are currently in the session and then, with your explicit confirmation, remove them.  

Would you like me to list the current players first, and then confirm removal of each (or all) of them?

> no just remove all

Agent: Sure thing. To permanently delete the player(s), I need a clear confirmation. Please reply with **confirm='yes'** to remove all current players (currently Alfred Winkleberry).

> yes

Agent: All player characters have been permanently removed from the session, and the action has been logged.

> list all players

Agent: There are currently **no players** in the session.

Conversation history cleared. Session data unchanged.

> create a new character named Timothy Quickfingers who's a bard from Eldenwood Villa

Agent: Sure! I can add Timothy Quickfingers to the party.

I have his name and background (bard from Eldenwood Villa).  
To create his character sheet I still need his starting numeric stats:

- **Health** (starting hit points)  
- **Defense** (base defensive rating)  
- **Damage** (base damage rating)

Would you like to set any additional stats (e.g., Mana, Initiative, etc.) as well, or just the three core values above? Let me know the numbers you’d like for Health, Defense, and Damage (and any extra stats if desired), and I’ll create Timothy for you.

> give him 20 Health, 1 Defense, 6 Damage

Agent: ✅ **Timothy Quickfingers** has been added to the session.

- **Class/Origin:** Bard from Eldenwood Villa  
- **Health:** 20  
- **Defense:** 1  
- **Damage:** 6  

If you’d like to set any additional attributes (Mana, Initiative, etc.) or give him starting equipment, just let me know!

> Give him a skill called "Fastest Fingers of the West" that gives him +2 Initiative

Agent: I can add the new skill, but I need a couple of details first:

- **Cooldown:** How many turns must Timothy wait after using “Fastest Fingers of the West” before he can use it again?
- **Cost:** Does the skill require any resource to activate (e.g., “none”, “5 mana”, etc.)?

Once I have those values, I’ll add the skill to Timothy’s character sheet.

> Cooldown: 1, Cost: None

Agent: Timothy Quickfingers now has the skill **“Fastest Fingers of the West”**:

- **Effect:** +2 Initiative  
- **Cooldown:** 1 turn  
- **Cost:** none  

The skill has been added to his character sheet and logged in the session notes. Let me know if you’d like to add equipment, other abilities, or any other updates!

Conversation history cleared. Session data unchanged.

> query Timothy Quickfingers

Agent: **Timothy Quickfingers**  
- **Health:** 20 (alive)  
- **Defense:** 1  
- **Damage:** 6  
- **Origin:** Bard from Eldenwood Villa  

**Inventory:** *(empty)*  

**Status Effects:** *(none)*  

**Skills:**  
- *Fastest Fingers of the West* – +2 Initiative (cooldown: 1 turn, cost: none)  

Session saved. Goodbye.